# FinGPT Two-Agent Signal Pipeline (Colab)

## Cell 1 — GPU check & install

This cell verifies GPU availability and installs notebook dependencies for the full pipeline demo.

In [ ]:
# Check GPU
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install dependencies
!pip install -q transformers accelerate alpaca-trade-api yfinance pydantic python-dotenv tqdm

## Cell 2 — Clone repo and set path

This cell clones the repository in Colab and sets the working directory so project imports resolve correctly.

In [ ]:
!git clone https://github.com/YOUR_REPO_URL /content/FinGPT_Part2
import sys, os
sys.path.insert(0, '/content/FinGPT_Part2')
os.chdir('/content/FinGPT_Part2')
print('Working directory:', os.getcwd())

## Cell 3 — Load secrets

This cell loads Alpaca credentials from Colab Secrets.

In [ ]:
import os
from google.colab import userdata

os.environ["ALPACA_API_KEY"] = userdata.get("ALPACA_API_KEY")
os.environ["ALPACA_API_SECRET"] = userdata.get("ALPACA_API_SECRET")

# Ensure model path uses HuggingFace Hub model ID in Colab
os.environ["FINGPT_MODEL_PATH"] = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# Enable shared single-LLM mode for Agent 1 + Agent 2
os.environ["SHARE_SINGLE_LLM_BETWEEN_AGENTS"] = "true"

print("Colab secrets loaded and single-LLM mode configured.")

## Cell 4 — Load one shared model and check VRAM

This cell loads a single shared local HuggingFace model instance used by both Agent 1 and Agent 2, then reports GPU memory before/after loading.

In [ ]:
import torch

def used_vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1e9

print(f"VRAM before model load: {used_vram_gb():.2f} GB")

from agent1 import extractor as agent1_extractor
from agent2 import reasoner as agent2_reasoner

# Load the shared LLM once via Agent 1
agent1_extractor._load_model()
used_after_load = used_vram_gb()
print(f"VRAM after shared model load: {used_after_load:.2f} GB")

# Confirm Agent 2 reuses the same loaded model instance
agent2_reasoner._load_model()
used_after_agent2 = used_vram_gb()
print(f"VRAM after Agent 2 init (shared): {used_after_agent2:.2f} GB")

total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
fits_device = used_after_agent2 < total_vram_gb
fits_40gb = used_after_agent2 <= 40.0

print(f"Shared model ready. Fits current GPU capacity ({total_vram_gb:.1f} GB): {fits_device}")
print(f"Shared model fits within 40 GB budget: {fits_40gb}")

## Cell 5 — Fetch articles

This cell fetches 10 recent articles for a fixed ticker set and shows a preview table.

In [ ]:
import pandas as pd
from ingestion.news_fetcher import fetch_recent_articles

TICKERS = ["AAPL", "NVDA", "TSLA"]
articles = fetch_recent_articles(TICKERS, limit=10)

preview_df = pd.DataFrame(articles)
preview_cols = [c for c in ["headline", "source", "created_at", "summary"] if c in preview_df.columns]
display(preview_df[preview_cols].head(10))
print(f"Fetched {len(articles)} article(s)")

## Cell 6 — Run Agent 1

This cell runs fingerprint extraction on each fetched article and prints per-article status. It also pretty-prints the first successful fingerprint.

In [ ]:
import json
from tqdm.auto import tqdm
from agent1.extractor import extract_fingerprint

fingerprints = []
first_successful_fp = None

for article in tqdm(articles, desc="Agent 1 extraction"):
    article_text = f"{article.get('headline', '')} {article.get('summary', '')}".strip()
    fp = extract_fingerprint(article_text)
    status = "OK" if fp is not None else "SKIPPED"
    print(f"{status} | {article.get('headline', '')[:90]}")
    if fp is not None:
        row = {
            "article": article,
            "fingerprint": fp,
        }
        fingerprints.append(row)
        if first_successful_fp is None:
            first_successful_fp = fp

print(f"\nExtracted {len(fingerprints)} fingerprint(s) out of {len(articles)} articles")
if first_successful_fp is not None:
    print("\nFirst successful NewsFingerprint:")
    print(json.dumps(first_successful_fp.model_dump(), indent=2))

## Cell 7 — Run Agent 2

This cell runs signal generation from extracted fingerprints and prints per-item status. It also pretty-prints the first successful signal.

In [ ]:
from tqdm.auto import tqdm
from agent2.reasoner import generate_signal

signals = []
first_successful_signal = None

for row in tqdm(fingerprints, desc="Agent 2 reasoning"):
    fp = row["fingerprint"]
    signal = generate_signal(fp)
    status = "OK" if signal is not None else "SKIPPED"
    print(f"{status} | {fp.headline[:90]}")
    if signal is not None:
        enriched = {
            "article": row["article"],
            "fingerprint": fp,
            "signal": signal,
        }
        signals.append(enriched)
        if first_successful_signal is None:
            first_successful_signal = signal

print(f"\nGenerated {len(signals)} signal(s) out of {len(fingerprints)} fingerprints")
if first_successful_signal is not None:
    import json
    print("\nFirst successful TradingSignal:")
    print(json.dumps(first_successful_signal.model_dump(), indent=2))

## Cell 8 — Results DataFrame

This cell builds a dataframe of signal + sentiment fields for quick inspection.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

rows = []
for row in tqdm(signals, desc="Building results table"):
    fp = row["fingerprint"]
    sig = row["signal"]
    rows.append(
        {
            "ticker": sig.ticker,
            "direction": sig.direction,
            "strategy_tag": sig.strategy_tag,
            "sentiment_score": fp.sentiment_score,
            "sentiment_confidence": fp.sentiment_confidence,
            "signal_confidence": sig.confidence,
            "cot": sig.cot,
        }
    )

results_df = pd.DataFrame(rows)
if results_df.empty:
    print("No signals produced.")
else:
    display(results_df)

## Cell 9 — Evaluation (mini)

This cell runs self-consistency evaluation on the fetched articles with N=3 runs and prints a per-article consistency table.

In [ ]:
from evaluation.evaluator import evaluate_self_consistency

article_texts = [f"{a.get('headline', '')} {a.get('summary', '')}".strip() for a in articles]
mini_eval = evaluate_self_consistency(article_texts, n_runs=3)

print(f"Consistency rate: {mini_eval['consistency_rate']:.3f}")
consistency_df = pd.DataFrame(mini_eval["per_article"])
display(consistency_df)

## Cell 10 — Save outputs

This cell saves the signal records to JSON for later analysis.

In [ ]:
import json
import os
from datetime import datetime, timezone
from tqdm.auto import tqdm

os.makedirs("output", exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
outpath = f"output/signals_demo_{timestamp}.json"

serializable_signals = []
for row in tqdm(signals, desc="Serializing outputs"):
    serializable_signals.append(
        {
            "article": row["article"],
            "fingerprint": row["fingerprint"].model_dump(),
            "signal": row["signal"].model_dump(),
        }
    )

with open(outpath, "w", encoding="utf-8") as handle:
    json.dump(serializable_signals, handle, indent=2)

print(f"Saved {len(serializable_signals)} records to {outpath}")